### Setting up envs and imports

In [1]:
from dotenv import load_dotenv
import os
from pageindex import PageIndexClient
import pageindex.utils as utils

load_dotenv()
PAGE_INDEX_API_KEY = os.getenv("API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
pi_client = PageIndexClient(api_key="PAGE_INDEX_API_KEY")
# print(pi_client)

if not PAGE_INDEX_API_KEY:
    print(
        "❌ Validation Failed: No 'API_KEY' entry detected within your local .env configuration."
    )
    exit(1)

clean_key = PAGE_INDEX_API_KEY.strip().replace('"', "").replace("'", "")
pi_client = PageIndexClient(api_key=clean_key)



### Testing Groq API KEY

In [4]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "What is the Capital of India?"}],
    reasoning_effort="medium",
)

print(completion.choices[0].message.content)


The capital of India is **New Delhi**.


### Loading PDF to create doc_id Of the PDF

In [5]:
# from pypdf import PdfReader
# reader = PdfReader(pdf_path)
pdf_path = "data/Policy-Document_LIC-s_New-Jeevan_Amar.pdf"
response = pi_client.submit_document(pdf_path)
doc_id = response["doc_id"]
print("Document Submitted:", doc_id)

# Health Check
status = pi_client.get_document(doc_id)["status"]
if status == "completed":
    print("Document processing completed")

Document Submitted: pi-cmqx9ddd5018b01pe8ykrkbqi


### Printing the Node Tree 

In [9]:
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    print("Simplified Tree Structure of the Document:")
    utils.print_tree(tree)
else:
    print("Processing document, please try again later...")

Simplified Tree Structure of the Document:
[{'title': "LIC's New Jeevan Amar (UIN:512N350V01)",
  'node_id': '0000',
  'summary': "# LIC's New Jeevan Amar (UIN:512N350V01)..."},
 {'title': 'PART-A', 'node_id': '0001', 'summary': 'This document serves as a formal cover l...'},
 {'title': 'Free Look Period',
  'node_id': '0002',
  'summary': "This document outlines the policyholder'..."},
 {'title': 'PREAMBLE', 'node_id': '0003', 'summary': 'This document serves as the preamble for...'},
 {'title': 'SCHEDULE', 'node_id': '0004', 'summary': 'This document is a policy schedule for L...'},
 {'title': 'PART– B: DEFINITIONS',
  'node_id': '0005',
  'summary': 'This document provides a comprehensive g...'},
 {'title': 'PART– C: BENEFITS',
  'node_id': '0006',
  'prefix_summary': 'This document outlines the benefits, con...',
  'nodes': [{'title': 'PART E', 'node_id': '0007', 'summary': '## PART E\n\nNot Applicable.\n'},
            {'title': 'PART – F: OTHER TERMS AND CONDITIONS',
            

### MultiPurpose Function for LLM Calling

In [10]:
from groq import Groq


def call_llm(prompt: str):
    client = Groq(api_key=GROQ_API_KEY)

    completion = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="medium",
        temperature=0,
        # response_format={"type": "json_object"},
    )

    return completion.choices[0].message.content



### Generating the JSON Tree

In [11]:
import json

query = "What are the conclusions in this document?"

tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

tree_search_result = call_llm(search_prompt)
print(type(tree_search_result))
print(tree_search_result)

# result = json.loads(tree_search_result)
# print(json.dumps(result, indent=4))


<class 'str'>
{
    "thinking": "The question asks for the conclusions of the document. Conclusions are typically found in the final sections of a policy document, often under 'Other Terms and Conditions' or the statutory provisions that wrap up the contract. In the provided tree, the last top‑level nodes are PART – F (node_id 0008) and PART – G (node_id 0009) along with its child sections (0010‑0014). These sections are positioned at the end of the document and are most likely to contain concluding statements or summary provisions. Therefore, the nodes most likely to hold the conclusions are 0008, 0009, and the child nodes 0010, 0011, 0012, 0013, and 0014.",
    "node_list": ["0008", "0009", "0010", "0011", "0012", "0013", "0014"]
}


In [12]:
import pageindex.utils as utils

node_map = utils.create_node_mapping(tree)
tree_search_result_json = json.loads(tree_search_result)

# print(type(node_map))
print(json.dumps(node_map, indent=4))

print("Reasoning Process:")
utils.print_wrapped(tree_search_result_json["thinking"])

print("\nRetrieved Nodes:")
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(
        f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}"
    )


{
    "0000": {
        "title": "LIC's New Jeevan Amar (UIN:512N350V01)",
        "node_id": "0000",
        "page_index": 1,
        "summary": "# LIC's New Jeevan Amar (UIN:512N350V01)\n**( A Non-Linked, Non-Participating, Individual, Pure Risk Premium Life Insurance Plan)**\n",
        "text": "# LIC's New Jeevan Amar (UIN:512N350V01)\n**( A Non-Linked, Non-Participating, Individual, Pure Risk Premium Life Insurance Plan)**\n"
    },
    "0001": {
        "title": "PART-A",
        "node_id": "0001",
        "page_index": 1,
        "summary": "This document serves as a formal cover letter for an insurance policy, instructing the policyholder to review their policy schedule, benefits, and available riders, while emphasizing the necessity of following prescribed procedures and timelines for exercising any plan options.",
        "text": "# PART-A\n\nRef: NB\n\n(Address and e-mail id of Branch Office)\n\nSpace for Name and Address of Policyholder\n\nSpace for Address and e-mail id of

### Answer Generartion from retrieved Context

In [13]:
if tree_search_result is None:
	raise ValueError("tree_search_result is None")

node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print("Retrieved Context:\n")
utils.print_wrapped(relevant_content[:1000] + "...")

answer_prompt = f"""
Answer the question based on the context below.

Question: {query}
Context: {relevant_content}

Instructions:
- Imagine you are explaining this to a friend who just bought an insurance policy
- Use "you" and "your" instead of "the policyholder" or "the insured"
- Replace legal terms with plain words, e.g.:
    * "jurisdiction" → "will handle your case"
    * "repudiation" → "rejection of your claim"
    * "bona fide" → "genuine"
    * "statutory provisions" → "legal rules"
- Keep bullet points but make each one a complete, friendly sentence
- Start with a plain 1-sentence summary
- End with a "Bottom line:" that tells the reader what they should actually DO or KNOW
- Do not use bold headers for every point — only bold the most critical warnings
- Write at a reading level suitable for someone with a high school education
- Avoid all Latin phrases and insurance jargon
"""

print("\n\n\nGenerated Answer:\n")

answer = call_llm(answer_prompt)
utils.print_wrapped(answer)

Retrieved Context:

## PART – F: OTHER TERMS AND CONDITIONS


## PART – G: STATUTORY PROVISIONS


### Section 45 of the Insurance Act 1938:

The provisions of Section 45 of the Insurance Act 1938, as amended from time to time, shall be
applicable. The current provisions are contained in Annexure-3 of this Policy Document.


### Grievance Redressal Mechanism:

#### Of the Corporation:

The Corporation has Grievance Redressal Officers at Branch/ Divisional/ Zonal/ Central Office to
redress grievances of customers. For ensuring quick redressal of customer grievances the Corporation
has introduced Customer friendly Integrated Complaint Management System through our Customer Portal
(website) which is http://www.licindia.in, where a registered policy holder can directly register
complaint/ grievance and track its status. Customers can also contact at e-mail id
co_complaints@licindia.com for redressal of any grievances.

Claimants not satisfied with the decision of death claim repudiation hav